# 01 — Carregar as bases da Receita Federal

**Objetivo:** ler os estabelecimentos (só os do município), as empresas e o Simples/MEI (só os CNPJs básicos desses estabelecimentos).

**Entradas:** `01_dados_brutos/receita_federal/` (`*.ESTABELE`, `*.EMPRECSV`, `*SIMPLES*`).

**Saídas:** `03_processamento/01_carregar_receita/`: `df_estab`, `df_empresas`, `df_simples` (Parquet + manifesto).

**Como rodar:** de cima para baixo. Com o cache válido, carrega em segundos; sem ele, a leitura dos ~20 GB leva bastante tempo (o log mostra o andamento). Para forçar a releitura, use `parametros.cache.forcar_reprocessar` no `config.yaml`.

**Seções:** 1. Estabelecimentos · 2. Empresas · 3. Simples Nacional / MEI · 4. Validação · 5. Resumo

In [ ]:
# ===================================================================
# PREPARAÇÃO: bibliotecas, config.yaml, log e entradas
# ===================================================================
# Bibliotecas que já vêm com o Python.
import datetime          # datas e horas
import importlib.util    # conferir se uma biblioteca está instalada, sem carregá-la
import json              # ler e gravar arquivos JSON
import os                # trocar arquivos de lugar com segurança
import re                # achar trechos de texto por padrão
import sys               # deixar o Python enxergar a pasta scripts/src
import time              # medir quanto tempo cada etapa leva
from pathlib import Path  # caminhos de pastas e arquivos

# Bibliotecas de cálculos e de tabelas (DataFrames).
import numpy as np
import pandas as pd

# Acha a raiz do projeto (a pasta do config.yaml), subindo a partir da pasta atual,
# e deixa o Python enxergar as funções comuns do projeto (scripts/src).
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.yaml").exists())
if str(RAIZ / "scripts") not in sys.path:
    sys.path.insert(0, str(RAIZ / "scripts"))

# Funções comuns: config.yaml, log, validação, arquivos de outras etapas.
from src import config, fontes, registro, validacao
from src.registro import formatar_tempo

# Lê o config.yaml e inicia o log (na tela e em 03_processamento/logs).
cfg = config.carregar(RAIZ)
log = registro.iniciar_log("01_carregar_receita", cfg)
# Hora de início do notebook (para o tempo total no resumo final).
inicio_notebook = time.time()
log.info("Preparando o notebook...")

# ===== PARÂMETROS (config.yaml) =====
PASTA_RECEITA = cfg.caminho("brutos.receita_federal")      # pasta com os arquivos da Receita
COD_MUNICIPIO_RF = cfg.projeto("cod_municipio_rf")          # código do município na Receita
CHUNK = cfg.parametro("leitura.chunk")                      # linhas lidas por vez
SEP_RECEITA = cfg.parametro("receita.separador")            # separador de colunas
ENC_RECEITA = cfg.parametro("receita.codificacao")          # codificação do texto
PADRAO_ESTAB = cfg.parametro("receita.padrao_estab")        # "*" = qualquer texto no nome do arquivo
PADRAO_EMPRESAS = cfg.parametro("receita.padrao_empresas")
PADRAO_SIMPLES = cfg.parametro("receita.padrao_simples")
COLUNAS_ESTAB = cfg.parametro("receita.colunas_estab")      # os arquivos da Receita não têm linha de títulos
COLUNAS_EMPRESAS = cfg.parametro("receita.colunas_empresas")
COLUNAS_SIMPLES = cfg.parametro("receita.colunas_simples")

# ===== CACHE (Parquet + manifesto, na pasta 03_processamento/01_carregar_receita) =====
from src.cache import CacheTabelas, assinatura_valores, impressao_arquivos


def descrever_cache(nome):
    """Devolve (origem, parâmetros, dependências) de cada tabela: é isso que decide se o cache vale."""
    # Parâmetros de leitura dos arquivos da Receita (iguais nas três tabelas).
    leitura_receita = {"separador": SEP_RECEITA, "codificacao": ENC_RECEITA}
    if nome == "df_estab":
        return (impressao_arquivos(PASTA_RECEITA, PADRAO_ESTAB),
                {**leitura_receita, "cod_municipio_rf": COD_MUNICIPIO_RF, "colunas": COLUNAS_ESTAB}, None)
    if nome in ("df_empresas", "df_simples"):
        # Empresas e Simples são filtrados pelos CNPJs básicos do df_estab: dependem dele.
        dep = {"df_estab_cnpjs_basicos": assinatura_valores(df_estab["cnpj_basico"])}
        padrao = PADRAO_EMPRESAS if nome == "df_empresas" else PADRAO_SIMPLES
        colunas = COLUNAS_EMPRESAS if nome == "df_empresas" else COLUNAS_SIMPLES
        return impressao_arquivos(PASTA_RECEITA, padrao), {**leitura_receita, "colunas": colunas}, dep
    raise KeyError(f"Tabela sem descrição de cache: {nome}")


cache = CacheTabelas(
    pasta=cfg.pasta_etapa("01_carregar_receita"),
    versoes=cfg.parametro("cache.versao_logica"),
    descrever=descrever_cache,
    forcar=cfg.parametro("cache.forcar_reprocessar", []),
    idade_maxima_dias=cfg.parametro("cache.idade_maxima_dias", {}),
)
# Mesmos nomes de antes, para as seções continuarem iguais.
verificar_cache, carregar_cache, salvar_cache = cache.verificar, cache.carregar, cache.salvar

# ===== ENTRADAS =====
log.info(f"Pasta da Receita: {PASTA_RECEITA} ({'existe' if PASTA_RECEITA.is_dir() else 'NÃO existe'})")
for padrao in (PADRAO_ESTAB, PADRAO_EMPRESAS, PADRAO_SIMPLES):
    log.info(f"    {padrao}: {len(impressao_arquivos(PASTA_RECEITA, padrao))} arquivo(s)")

log.info(f"Preparação concluída! ({formatar_tempo(time.time() - inicio_notebook)})")

## 1. Estabelecimentos
Lê todos os arquivos `.ESTABELE` em blocos, mantém só os estabelecimentos do município (código da Receita) e guarda o resultado em `df_estab`.
Se o cache de `df_estab` vale, carrega dele e não lê os arquivos.

In [ ]:
log.info("SEÇÃO 1: estabelecimentos (usa o cache quando ele vale)...")
inicio_b2 = time.time()

# Confere se o cache da tabela vale (arquivos de origem, parâmetros, versão e dependências iguais aos de quando foi gravado).
log.info("Conferindo o cache de df_estab...")
usar_cache, motivo_cache = verificar_cache("df_estab")
log.info("Conferência do cache concluída!")

if usar_cache:
    log.info(f"Usando o cache ({motivo_cache}): carregando df_estab...")
    df_estab = carregar_cache("df_estab")
    log.info(f"df_estab carregado do cache! {len(df_estab):,} linhas | {df_estab.shape[1]} colunas.")
else:
    log.info(f"Cache não usado ({motivo_cache}): lendo os arquivos originais...")
    log.info("Lendo estabelecimentos...")
    # Guarda a hora de início para calcular o tempo total da etapa no final.
    inicio_etapa = time.time()

    # Procura na pasta todos os arquivos do padrão e coloca em ordem alfabética.
    arquivos = sorted(Path(PASTA_RECEITA).glob(PADRAO_ESTAB))
    log.info(f"{len(arquivos)} arquivo(s) de estabelecimentos encontrado(s)!")

    # Lista vazia onde guardaremos os pedaços filtrados de cada arquivo.
    partes = []

    # Passa por cada arquivo, um de cada vez (numero = 1, 2, 3...).
    for numero, arquivo in enumerate(arquivos, start=1):
        log.info(f"[{numero}/{len(arquivos)}] Lendo {arquivo.name}...")
        inicio_arquivo = time.time()

        # Contadores de conferência: linhas lidas e linhas mantidas neste arquivo.
        total_arquivo = 0
        mantidas_arquivo = 0

        # Abre o arquivo para ler em blocos (chunksize), e não tudo de uma vez.
        # header=None: o arquivo não tem linha de títulos | names: damos os nomes das colunas
        # dtype=str: lê tudo como texto, sem perder zeros à esquerda
        leitor = pd.read_csv(
            arquivo,
            sep=SEP_RECEITA,
            header=None,
            names=COLUNAS_ESTAB,
            dtype=str,
            encoding=ENC_RECEITA,
            quotechar='"',
            chunksize=CHUNK,
            on_bad_lines="warn",
        )

        # Passa por cada bloco de linhas do arquivo.
        for bloco in leitor:
            # Soma as linhas deste bloco ao total lido do arquivo.
            total_arquivo += len(bloco)

            # Mantém só as linhas cujo código de município é o do nosso município.
            filtrado = bloco[bloco["cod_municipio_rf"] == COD_MUNICIPIO_RF]
            mantidas_arquivo += len(filtrado)

            # Se sobrou alguma linha neste bloco, guarda na lista.
            if not filtrado.empty:
                partes.append(filtrado)

            # Mostra o andamento (flush=True faz a mensagem aparecer na hora).
            log.info(f"    {total_arquivo:,} linhas lidas, {mantidas_arquivo:,} mantidas...")

        log.info(f"[{numero}/{len(arquivos)}] {arquivo.name} concluído! "
              f"{mantidas_arquivo:,} de {total_arquivo:,} linhas mantidas "
              f"({formatar_tempo(time.time() - inicio_arquivo)}).")

    # Junta todos os pedaços em um único DataFrame.
    log.info("Juntando os pedaços em um único DataFrame (df_estab)...")
    if partes:
        # ignore_index=True refaz a numeração das linhas do zero (0, 1, 2...).
        df_estab = pd.concat(partes, ignore_index=True)
    else:
        # Se nada foi mantido, cria um DataFrame vazio só com os nomes das colunas.
        df_estab = pd.DataFrame(columns=COLUNAS_ESTAB)

    log.info(f"Leitura concluída! df_estab: {len(df_estab):,} linhas | "
          f"{df_estab.shape[1]} colunas | tempo total: {formatar_tempo(time.time() - inicio_etapa)}.")

    # Grava o cache para as próximas sessões.
    salvar_cache("df_estab", df_estab)

log.info(f"\nSEÇÃO 1 concluída! Tempo total: {formatar_tempo(time.time() - inicio_b2)}.")

## 2. Empresas
Lê todos os arquivos `.EMPRECSV`, mantém só as empresas cujo `cnpj_basico` está no `df_estab` e guarda o resultado em `df_empresas`.
Se o cache vale (inclusive com o mesmo conjunto de CNPJs básicos do `df_estab`), carrega dele.

In [ ]:
log.info("SEÇÃO 2: empresas (usa o cache quando ele vale)...")
inicio_b3 = time.time()

# Confere se o cache da tabela vale (arquivos de origem, parâmetros, versão e dependências iguais aos de quando foi gravado).
log.info("Conferindo o cache de df_empresas...")
usar_cache, motivo_cache = verificar_cache("df_empresas")
log.info("Conferência do cache concluída!")

if usar_cache:
    log.info(f"Usando o cache ({motivo_cache}): carregando df_empresas...")
    df_empresas = carregar_cache("df_empresas")
    log.info(f"df_empresas carregado do cache! {len(df_empresas):,} linhas | {df_empresas.shape[1]} colunas.")
else:
    log.info(f"Cache não usado ({motivo_cache}): lendo os arquivos originais...")
    log.info("Lendo empresas...")
    inicio_etapa = time.time()

    # Monta o conjunto de cnpj_basico (8 primeiros dígitos do CNPJ) presentes no df_estab.
    # set() elimina repetições e torna a busca muito mais rápida. O df_estab só é LIDO, não alterado.
    log.info("Montando a lista de CNPJs básicos do município...")
    cnpjs_do_municipio = set(df_estab["cnpj_basico"])
    log.info(f"Lista montada! {len(cnpjs_do_municipio):,} CNPJs básicos distintos.")

    # Procura os arquivos do padrão na pasta, em ordem alfabética.
    arquivos = sorted(Path(PASTA_RECEITA).glob(PADRAO_EMPRESAS))
    log.info(f"{len(arquivos)} arquivo(s) de empresas encontrado(s)!")

    partes = []

    for numero, arquivo in enumerate(arquivos, start=1):
        log.info(f"[{numero}/{len(arquivos)}] Lendo {arquivo.name}...")
        inicio_arquivo = time.time()
        total_arquivo = 0
        mantidas_arquivo = 0

        # Mesma leitura em blocos das etapas anteriores, agora com as colunas de empresas.
        leitor = pd.read_csv(
            arquivo,
            sep=SEP_RECEITA,
            header=None,
            names=COLUNAS_EMPRESAS,
            dtype=str,
            encoding=ENC_RECEITA,
            quotechar='"',
            chunksize=CHUNK,
        )

        for bloco in leitor:
            total_arquivo += len(bloco)

            # isin() marca como verdadeira cada linha cujo cnpj_basico está no conjunto;
            # bloco[...] fica só com as verdadeiras.
            filtrado = bloco[bloco["cnpj_basico"].isin(cnpjs_do_municipio)]
            mantidas_arquivo += len(filtrado)

            if not filtrado.empty:
                partes.append(filtrado)

            log.info(f"    {total_arquivo:,} linhas lidas, {mantidas_arquivo:,} mantidas...")

        log.info(f"[{numero}/{len(arquivos)}] {arquivo.name} concluído! "
              f"{mantidas_arquivo:,} de {total_arquivo:,} linhas mantidas "
              f"({formatar_tempo(time.time() - inicio_arquivo)}).")

    log.info("Juntando os pedaços em um único DataFrame (df_empresas)...")
    if partes:
        df_empresas = pd.concat(partes, ignore_index=True)
    else:
        df_empresas = pd.DataFrame(columns=COLUNAS_EMPRESAS)

    log.info(f"Leitura concluída! df_empresas: {len(df_empresas):,} linhas | "
          f"{df_empresas.shape[1]} colunas | tempo total: {formatar_tempo(time.time() - inicio_etapa)}.")

    # Grava o cache para as próximas sessões.
    salvar_cache("df_empresas", df_empresas)

log.info(f"\nSEÇÃO 2 concluída! Tempo total: {formatar_tempo(time.time() - inicio_b3)}.")

## 3. Simples Nacional / MEI
Lê o arquivo do Simples, mantém só os CNPJs básicos do `df_estab` e guarda o resultado em `df_simples`.
Se o cache vale (inclusive com o mesmo conjunto de CNPJs básicos do `df_estab`), carrega dele.

In [ ]:
log.info("SEÇÃO 3: Simples Nacional / MEI (usa o cache quando ele vale)...")
inicio_b4 = time.time()

# Confere se o cache da tabela vale (arquivos de origem, parâmetros, versão e dependências iguais aos de quando foi gravado).
log.info("Conferindo o cache de df_simples...")
usar_cache, motivo_cache = verificar_cache("df_simples")
log.info("Conferência do cache concluída!")

if usar_cache:
    log.info(f"Usando o cache ({motivo_cache}): carregando df_simples...")
    df_simples = carregar_cache("df_simples")
    log.info(f"df_simples carregado do cache! {len(df_simples):,} linhas | {df_simples.shape[1]} colunas.")
else:
    log.info(f"Cache não usado ({motivo_cache}): lendo os arquivos originais...")
    log.info("Lendo Simples Nacional / MEI...")
    inicio_etapa = time.time()

    # Monta novamente o conjunto de cnpj_basico do município (o df_estab só é lido, não alterado).
    log.info("Montando a lista de CNPJs básicos do município...")
    cnpjs_do_municipio = set(df_estab["cnpj_basico"])
    log.info(f"Lista montada! {len(cnpjs_do_municipio):,} CNPJs básicos distintos.")

    # Procura os arquivos do padrão; is_file() garante que só entrem arquivos (e não pastas).
    arquivos = sorted(p for p in Path(PASTA_RECEITA).glob(PADRAO_SIMPLES) if p.is_file())
    log.info(f"{len(arquivos)} arquivo(s) do Simples encontrado(s)!")

    partes = []

    for numero, arquivo in enumerate(arquivos, start=1):
        log.info(f"[{numero}/{len(arquivos)}] Lendo {arquivo.name}...")
        inicio_arquivo = time.time()
        total_arquivo = 0
        mantidas_arquivo = 0

        leitor = pd.read_csv(
            arquivo,
            sep=SEP_RECEITA,
            header=None,
            names=COLUNAS_SIMPLES,
            dtype=str,
            encoding=ENC_RECEITA,
            quotechar='"',
            chunksize=CHUNK,
        )

        for bloco in leitor:
            total_arquivo += len(bloco)

            # Mantém só as linhas cujo cnpj_basico está no conjunto do município.
            filtrado = bloco[bloco["cnpj_basico"].isin(cnpjs_do_municipio)]
            mantidas_arquivo += len(filtrado)

            if not filtrado.empty:
                partes.append(filtrado)

            log.info(f"    {total_arquivo:,} linhas lidas, {mantidas_arquivo:,} mantidas...")

        log.info(f"[{numero}/{len(arquivos)}] {arquivo.name} concluído! "
              f"{mantidas_arquivo:,} de {total_arquivo:,} linhas mantidas "
              f"({formatar_tempo(time.time() - inicio_arquivo)}).")

    log.info("Juntando os pedaços em um único DataFrame (df_simples)...")
    if partes:
        df_simples = pd.concat(partes, ignore_index=True)
    else:
        df_simples = pd.DataFrame(columns=COLUNAS_SIMPLES)

    log.info(f"Leitura concluída! df_simples: {len(df_simples):,} linhas | "
          f"{df_simples.shape[1]} colunas | tempo total: {formatar_tempo(time.time() - inicio_etapa)}.")

    # Grava o cache para as próximas sessões.
    salvar_cache("df_simples", df_simples)

log.info(f"\nSEÇÃO 3 concluída! Tempo total: {formatar_tempo(time.time() - inicio_b4)}.")

## 4. Validação
Linhas, nulos e duplicados de cada tabela, e se todo estabelecimento tem a sua empresa.

In [ ]:
log.info("SEÇÃO 4: validação das tabelas...")
validacao.validar_tabela("df_estab", df_estab, chave=["cnpj_basico", "cnpj_ordem", "cnpj_dv"],
                         obrigatorias=["cnpj_basico", "cnpj_ordem", "cnpj_dv"])
validacao.validar_tabela("df_empresas", df_empresas, chave="cnpj_basico", obrigatorias=["cnpj_basico"])
validacao.validar_tabela("df_simples", df_simples, chave="cnpj_basico", obrigatorias=["cnpj_basico"])
# Todo estabelecimento deve ter a sua empresa (mesmo CNPJ básico).
sem_empresa = int((~df_estab["cnpj_basico"].isin(df_empresas["cnpj_basico"])).sum())
(log.info if sem_empresa == 0 else log.warning)(f"Estabelecimentos sem empresa correspondente: {sem_empresa:,}")
log.info("SEÇÃO 4 concluída!")

## 5. Resumo
Números principais, arquivos gerados e tempo total.

In [ ]:
log.info("RESUMO do 01_carregar_receita...")
log.info(f"Estabelecimentos do município: {len(df_estab):,} | empresas: {len(df_empresas):,} | Simples/MEI: {len(df_simples):,}")
log.info("Arquivos gerados:")
for nome_tabela in ['df_estab', 'df_empresas', 'df_simples']:
    caminho = cfg.caminho("processamento." + nome_tabela)
    log.info(f"    {caminho.relative_to(RAIZ)} ({caminho.stat().st_size / 1e6:.1f} MB)")
log.info(f"RESUMO concluído! Tempo total do notebook: {formatar_tempo(time.time() - inicio_notebook)}.")